# Mrk 509: a single-object distance duality test

Run from `horizon/` using the `.venv/bin/python` kernel.

Mrk 509 is the only quasar with both a SARM angular diameter distance
(Li et al. 2025, Table 2) and UV and X-ray fluxes in the UV--X-ray catalogue
(Lusso et al. 2020, where it is one of the 13 local AGN). Here we combine the
two to measure $\eta = d_L / [(1+z)^2 d_A]$.

## Import packages

In [1]:
import numpy as np
import pandas as pd

## Luminosity distance
The Lusso et al. (2020) table gives a distance modulus for each quasar, with
the UV--X-ray intercept cross-calibrated on Pantheon supernovae assuming
$H_0 = 70$ km/s/Mpc. Its uncertainty `e_DM` includes the flux and calibration
errors but not the intrinsic scatter of the relation, so we add the intrinsic
scatter `delta_dex` separately, as in the forecast.

In [2]:
columns = ['name', 'ra', 'dec', 'z', 'logFUV', 'e_logFUV', 'logFX', 'e_logFX',
           'group', 'gammax', 'e_gammax', 'DM', 'e_DM']
lusso = pd.read_csv('data/lusso2020_table3.dat', sep=r'\s+', names=columns)
mrk509 = lusso.set_index('name').loc['MRK509']

uvx = {'gamma': 0.60,
       'delta_dex': 0.21,
       'H0_calibration': 70.0}

z = mrk509['z']
d_L = 10**(mrk509['DM'] / 5 - 5)  # Mpc
sigma_ln_dl = np.hypot(np.log(10) / 5 * mrk509['e_DM'],
                       np.log(10) * uvx['delta_dex'] / (2 * (1 - uvx['gamma'])))
print(f'z = {z},  d_L = {d_L:.1f} Mpc,  sigma_ln_dL = {sigma_ln_dl:.2f}')

z = 0.0344,  d_L = 136.5 Mpc,  sigma_ln_dL = 0.61


## Angular diameter distance
From Table 2 of Li et al. (2025), $\log(d_A/{\rm Mpc}) = 2.41^{+0.22}_{-0.23}$.
We symmetrise the interval and convert it to natural log.

In [3]:
sarm = {'log_dA': 2.41,
        'err_up': 0.22,
        'err_down': 0.23}

d_A = 10**sarm['log_dA']  # Mpc
sigma_ln_da = np.log(10) * (sarm['err_up'] + sarm['err_down']) / 2
print(f'd_A = {d_A:.1f} Mpc,  sigma_ln_dA = {sigma_ln_da:.2f}')

d_A = 257.0 Mpc,  sigma_ln_dA = 0.52


## Measure $\eta$

In [4]:
ln_eta = np.log(d_L / ((1 + z)**2 * d_A))
sigma_ln_eta = np.hypot(sigma_ln_dl, sigma_ln_da)

print(f'ln eta = {ln_eta:.2f} ± {sigma_ln_eta:.2f}, '
      f'{abs(ln_eta) / sigma_ln_eta:.1f} sigma from eta = 1')
eta = np.exp(ln_eta)
eta_up = np.exp(ln_eta + sigma_ln_eta) - eta
eta_down = eta - np.exp(ln_eta - sigma_ln_eta)
print(f'eta = {eta:.2f} +{eta_up:.2f} -{eta_down:.2f}')

ln eta = -0.70 ± 0.80, 0.9 sigma from eta = 1
eta = 0.50 +0.61 -0.27


## Dependence on the calibration
$d_L$ scales as $1/H_0$ through the supernova calibration, whereas the SARM
$d_A$ does not, so $\eta$ inherits the calibration offset discussed in the
paper. Rescaled to the $H_0$ used for the mocks:

In [5]:
H0_mocks = 67.36

ln_eta_mocks = ln_eta + np.log(uvx['H0_calibration'] / H0_mocks)
print(f'ln eta = {ln_eta_mocks:.2f} ± {sigma_ln_eta:.2f} for H0 = {H0_mocks}')

ln eta = -0.66 ± 0.80 for H0 = 67.36
